# 02 — Group Study (multivariate Optuna ablation)

11축 동시 탐색 (TPE multivariate). LGBM HP는 default 고정 → 축 효과 + 상호작용만 학습.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*.csv`
- **출력**: `4_output/0_baseline/group/{optuna.db, trials.csv, param_importance.csv}`
- **참조**: [strategy.md §5](strategy.md), [strategy_common.md §4·§6·§8](../strategy_common.md)

## 1. 환경 설정 + 데이터 로드

In [ ]:
import os, sys
RESUME = True   # 기존 Optuna study(db)에 이어서 학습할지 (필요시 config 셀에서 덮어씀)
# ── Colab이면 코드 번들 1개(code.zip)만 받아 풀기 — 데이터·경로·폰트는 setup.py가 처리 ──
try:
    import google.colab  # Colab에서만 import 성공
    GDRIVE_CODE_ID = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip = setup.py+requirements+utils+2_preprocessing+3_modeling 지원코드
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip -q install gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
    os.chdir('/content/project')
except ImportError:
    pass
# ── 공통: cwd에서 위로 setup.py(+utils/)를 자동탐색해 실행 (노트북 깊이·드라이브 위치 무관) ──
_d = os.getcwd()
while not (os.path.exists(os.path.join(_d, 'setup.py')) and os.path.isdir(os.path.join(_d, 'utils'))):
    _p = os.path.dirname(_d)
    if _p == _d:
        raise RuntimeError('프로젝트 루트(setup.py + utils/)를 못 찾음 — cwd 확인')
    _d = _p
if _d not in sys.path:
    sys.path.insert(0, _d)
import runpy
runpy.run_path(os.path.join(_d, 'setup.py'))

from utils.config import PROJECT_ROOT

# 0_baseline 폴더를 경로에 추가 → `import axes` 가 이 노트북 옆의 axes.py를 찾게
BASELINE_DIR = os.path.join(PROJECT_ROOT, '3_modeling', '0_baseline')
if BASELINE_DIR not in sys.path:
    sys.path.insert(0, BASELINE_DIR)

import warnings
warnings.filterwarnings('ignore')

from utils.data import load_all, get_feat_cols, split_xs
import axes

xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
axes.set_data(xs, xs_dict, ys, feat_cols)   # run_one들이 공유할 데이터를 모듈에 1회 주입

print(f'Feature 수: {len(feat_cols)}')
print(f'Die 수: train={len(xs_dict["train"]):,}, val={len(xs_dict["validation"]):,}, test={len(xs_dict["test"]):,}')

## 2. Optuna study 설정

- Sampler: TPE (seed=None, multivariate=True, group=True) — strategy_common §4
- Pruner: 비활성 (5-fold 다 끝나야 점수 나옴)
- Storage: sqlite (4_output/0_baseline/group/optuna.db)
- Trial: default 300 (strategy.md §5.4)

In [ ]:
# 노트북 상단 단일 파라미터
N_JOBS = 5         # 모델 학습 병렬도
N_TRIALS = 300     # Optuna trial 수
TIMEOUT_SEC = None  # 초 단위, None=무제한 (Colab 타임아웃 대비 시 숫자로)
N_ESTIMATORS = 100 # LGBM n_estimators 고정 — HP가 아니라 전처리 축 효과만 본다

# group study용 축 = OAT 11축 그대로, 단 impute에서 'knn' 제외.
#   knn-impute는 fit당 ~70분(spatial/median의 20~25배) → 300 trial이 며칠 걸리고,
#   impute 축 marginal 효과는 ≈1e-6로 트리 모델엔 거의 무관. knn은 OAT에서 1셀로만 측정한다.
GROUP_AXES = {**axes.AXES, 'impute': ['spatial', 'median']}

import json
import optuna
from datetime import datetime
from optuna.samplers import TPESampler

OUT_DIR = os.path.join(PROJECT_ROOT, '4_output', '0_baseline', 'group')
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, 'optuna.db')
META_JSON = os.path.join(OUT_DIR, 'meta.json')
STORAGE_URL = f'sqlite:///{DB_PATH}'

# TPE: multivariate=축 간 결합 분포 학습, group=CLF=off 같은 조건부 축을 자동으로 건너뜀. seed=None → run마다 다양성
sampler = TPESampler(
    seed=None,
    multivariate=True,
    group=True,
)

study = optuna.create_study(
    study_name='baseline_group',
    storage=STORAGE_URL,
    sampler=sampler,
    direction='minimize',
    load_if_exists=True,   # 같은 db가 있으면 이어서 (끊김 후 재실행 대비)
)

# meta.json — 이번 run의 study 설정 + reference/축/고정 전처리 등을 박제
meta = {
    'created':       datetime.now().isoformat(timespec='seconds'),
    'n_jobs':        N_JOBS,
    'n_trials':      N_TRIALS,
    'n_estimators':  N_ESTIMATORS,
    'study_name':    'baseline_group',
    'sampler':       'TPESampler(seed=None, multivariate=True, group=True)',
    'pruner':        'None (5-fold complete eval)',
    'direction':     'minimize',
    'objective':     'oof_rmse',
    'reference':     axes.REFERENCE,
    'axes':          {k: list(map(str, v)) for k, v in GROUP_AXES.items()},
    'axes_note':     "impute에서 'knn' 제외 (fit당 ~70분 — OAT에서만 측정). 그 외는 axes.AXES와 동일",
    'agg_preset_lib': axes.AGG_PRESET_LIB,
    'pp_pin': {
        'cleaning': axes.PP_PIN_CLEANING,
        'outlier':  axes.PP_PIN_OUTLIER,
        'binarize': axes.PP_PIN_BINARIZE,
        'iso':      axes.PP_PIN_ISO,
        'lds':      axes.PP_PIN_LDS,
        'ge':       axes.PP_PIN_GE,
    },
    'exclude_cols':  axes.EXCLUDE_COLS,
}
with open(META_JSON, 'w') as f:
    json.dump(meta, f, indent=2, default=str, ensure_ascii=False)

print(f'Study : baseline_group')
print(f'Storage: {DB_PATH}')
print(f'기존 trial: {len(study.trials)}')
print(f'meta.json saved → {META_JSON}')


## 3. Objective + study.optimize

GROUP_AXES(=axes.AXES, 단 impute는 knn 제외)의 categorical을 trial.suggest_categorical로 샘플 → axes.run_one(cfg, seed=42) 호출. objective = OOF RMSE (train 5-fold CV).

In [ ]:
def objective(trial):
    # 전처리 축을 각각 categorical로 샘플 (GROUP_AXES = OAT 11축, 단 impute는 knn 제외)
    cfg = {
        axis: trial.suggest_categorical(axis, options)
        for axis, options in GROUP_AXES.items()
    }
    # 이 cfg로 전처리+LGBM 5-fold (HP는 default 고정, seed=42 고정 → 축 효과·상호작용만 학습)
    result = axes.run_one(
        cfg, seed=42, n_jobs=N_JOBS, n_estimators=N_ESTIMATORS,
    )
    # objective는 oof_rmse (train 5-fold CV) — strategy_common: best 탐색은 train OOF 기준.
    # val/test를 직접 최적화하면 cherry-picking이라 user_attr에 참고용으로만 기록.
    trial.set_user_attr('val_rmse',  result['val_rmse'])
    trial.set_user_attr('test_rmse', result['test_rmse'])
    trial.set_user_attr('elapsed_sec', result['elapsed_sec'])
    trial.set_user_attr('effective_target_transform', result['effective_target_transform'])  # tweedie loss면 'none' override 추적
    return result['oof_rmse']

# trial은 직렬(n_jobs=1)로 — 모델 내부 N_JOBS와 곱해져 코어가 과할당되는 걸 막음
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, n_jobs=1, show_progress_bar=True)

## 4. 산출물 저장

- `trials.csv` — study.trials_dataframe()
- `param_importance.csv` — fANOVA 기반 축 중요도 (OAT tornado와 비교용)

optuna.db는 storage로 자동 저장됨.

In [ ]:
import pandas as pd
from optuna.importance import get_param_importances, FanovaImportanceEvaluator

# 전체 trial 표를 csv로
trials_df = study.trials_dataframe()
trials_df.to_csv(os.path.join(OUT_DIR, 'trials.csv'), index=False)

# fANOVA 기반 축 중요도 — OAT tornado(축별 marginal)와 비교용
imp = get_param_importances(study, evaluator=FanovaImportanceEvaluator(seed=42))
imp_df = pd.DataFrame([{'axis': k, 'importance': v} for k, v in imp.items()])
imp_df.to_csv(os.path.join(OUT_DIR, 'param_importance.csv'), index=False)

print(f'trials.csv         : {len(trials_df)} rows → {OUT_DIR}/trials.csv')
print(f'param_importance   : {len(imp_df)} axes  → {OUT_DIR}/param_importance.csv')